# 02_kv_cache_and_serving_profiler: Autoregressive KV Cache Latency & VRAM Profiling
    
This notebook profiles the memory and computational latency of autoregressive text decoding. 

We will implement:
1. **Naive Autoregressive Decoding**: Recalculates all token projections at each generation step.
2. **KV Cache-Based Decoding**: Only projects the single new token and retrieves historical values.
3. **Prefill vs. Decode Latency Profiling**: Comparing speed (tokens/sec).
4. **VRAM Memory Footprint Profiling**: Scaling analysis for MHA vs. GQA vs. MQA.
5. **PagedAttention Page Table Manager Simulation**: Mocking OS-style paging in PyTorch.


## 1. Setup and Environment Initialization

In [1]:
import torch
import torch.nn as nn
import time
import numpy as np

# Set random seed
torch.manual_seed(42)

# Configurations mimicking standard LLM layers
B = 4           # Batch size
h = 32          # Query heads
d_k = 128       # Head dimension
d_model = h * d_k # 4096 hidden dimension


### Output Explanation: Setup
- **Config Constants**: We establish a model dimension of 4096 to emulate real 7B model sizes.


## 2. Autoregressive Decoding Loop (Naïve vs. KV Cache)

In [2]:
# Projection layers simulation
q_proj = nn.Linear(d_model, d_model, bias=False)
k_proj = nn.Linear(d_model, d_model, bias=False)
v_proj = nn.Linear(d_model, d_model, bias=False)

def generate_naive(prompt_tokens, num_tokens_to_generate):
    B, L, d = prompt_tokens.shape
    tokens = prompt_tokens
    
    start_time = time.time()
    for step in range(num_tokens_to_generate):
        # Naive approach: recalculate projections for the entire sequence at each step
        q = q_proj(tokens) # [B, SeqLen, d]
        k = k_proj(tokens) # [B, SeqLen, d]
        v = v_proj(tokens) # [B, SeqLen, d]
        
        # Calculate attention over the entire sequence length
        scores = torch.matmul(q, k.transpose(-2, -1)) / (d_k ** 0.5)
        # Select attention scores for the last token only (autoregressive decoding step)
        attn_weights = torch.softmax(scores[:, -1:, :], dim=-1)
        context = torch.matmul(attn_weights, v) # [B, 1, d]
        
        # Simulate next token selection & append
        next_token = context[:, -1:, :]
        tokens = torch.cat([tokens, next_token], dim=1)
        
    duration = time.time() - start_time
    return tokens, duration

def generate_with_kv_cache(prompt_tokens, num_tokens_to_generate):
    B, L, d = prompt_tokens.shape
    tokens = prompt_tokens
    
    # PREFILL PHASE: Process prompt tokens in parallel and initialize caches
    k_cache = k_proj(prompt_tokens) # [B, L, d]
    v_cache = v_proj(prompt_tokens) # [B, L, d]
    
    q_last = q_proj(prompt_tokens[:, -1:, :]) # [B, 1, d]
    scores = torch.matmul(q_last, k_cache.transpose(-2, -1)) / (d_k ** 0.5)
    attn_weights = torch.softmax(scores, dim=-1)
    next_token = torch.matmul(attn_weights, v_cache)
    
    tokens_out = torch.cat([prompt_tokens, next_token], dim=1)
    
    # DECODING PHASE: Run step-by-step appending only the new token
    start_time = time.time()
    for step in range(1, num_tokens_to_generate):
        x_new = tokens_out[:, -1:, :] # [B, 1, d]
        
        # Project only the single new token
        q_new = q_proj(x_new)
        k_new = k_proj(x_new)
        v_new = v_proj(x_new)
        
        # Append to caches
        k_cache = torch.cat([k_cache, k_new], dim=1)
        v_cache = torch.cat([v_cache, v_new], dim=1)
        
        # Attention on cached history
        scores = torch.matmul(q_new, k_cache.transpose(-2, -1)) / (d_k ** 0.5)
        attn_weights = torch.softmax(scores, dim=-1)
        next_token = torch.matmul(attn_weights, v_cache)
        
        tokens_out = torch.cat([tokens_out, next_token], dim=1)
        
    duration = time.time() - start_time
    return tokens_out, duration

# Sanity check run
prompt = torch.randn(B, 16, d_model)
naive_tokens, naive_time = generate_naive(prompt, 5)
cache_tokens, cache_time = generate_with_kv_cache(prompt, 5)

print("Naive generation time:", naive_time)
print("KV Cache generation time:", cache_time)
assert naive_tokens.shape == cache_tokens.shape, "Sequence output shapes mismatch!"
print("Generated shapes match successfully:", naive_tokens.shape)


Naive generation time: 0.1114652156829834
KV Cache generation time: 0.030636072158813477
Generated shapes match successfully: torch.Size([4, 21, 4096])


### Output Explanation: Generative Simulation
- **Output Equivalency**: Both loops generate identical sequence sizes.
- **Arithmetic Reduction**: The cache loop only projects a single token input at each generation step, skipping the $O(L)$ projection computation.


## 3. Prefill vs. Decoding Latency Profiling

In [3]:
# Profile latency over longer context length
prompt = torch.randn(B, 256, d_model) # Prompt length 256
gen_tokens = 50

_, naive_duration = generate_naive(prompt, gen_tokens)
_, cache_duration = generate_with_kv_cache(prompt, gen_tokens)

print(f"Latency Profiling (generating {gen_tokens} tokens on a prompt of size {prompt.shape[1]}):")
print(f"- Naive approach: {naive_duration:.4f} seconds ({gen_tokens / naive_duration:.2f} tokens/sec)")
print(f"- KV Cache approach: {cache_duration:.4f} seconds ({gen_tokens / cache_duration:.2f} tokens/sec)")
print(f"Speedup factor: {naive_duration / cache_duration:.2f}x")


Latency Profiling (generating 50 tokens on a prompt of size 256):
- Naive approach: 12.8202 seconds (3.90 tokens/sec)
- KV Cache approach: 1.0418 seconds (47.99 tokens/sec)
Speedup factor: 12.31x


### Output Explanation: Profiling Results
- **Generation Speed**: The KV Cache approach generates tokens significantly faster.
- **Scaling Effect**: As sequence length $L$ scales to thousands of tokens, the naive approach slows down quadratically, while the KV Cache approach maintains constant execution speed.


## 4. Attention Variant Memory Footprint Analysis (MHA vs. GQA vs. MQA)

In [4]:
# Calculate and print KV cache VRAM footprint in MB
B_val = 8
layers_val = 80
d_k_val = 128
h_q = 64

lengths = [512, 1024, 2048, 4096, 8192, 16384]
bytes_per_param = 2 # fp16/bf16

print("KV Cache Memory footprint comparison (MB) for Batch Size=8, Layers=80:")
print(f"{'Context Length L':<18} | {'MHA (64 heads)':<16} | {'GQA (8 heads)':<16} | {'MQA (1 head)':<16}")
print("-" * 72)
for L in lengths:
    # Formula: 2 (keys & values) * B * L * layers * h_kv * d_k * bytes_per_param / 1024^2
    mha_mb = (4 * B_val * L * layers_val * 64 * d_k_val * bytes_per_param) / (1024**2)
    gqa_mb = (4 * B_val * L * layers_val * 8 * d_k_val * bytes_per_param) / (1024**2)
    mqa_mb = (4 * B_val * L * layers_val * 1 * d_k_val * bytes_per_param) / (1024**2)
    print(f"{L:<18} | {mha_mb:<16.2f} | {gqa_mb:<16.2f} | {mqa_mb:<16.2f}")


KV Cache Memory footprint comparison (MB) for Batch Size=8, Layers=80:
Context Length L   | MHA (64 heads)   | GQA (8 heads)    | MQA (1 head)    
------------------------------------------------------------------------
512                | 20480.00         | 2560.00          | 320.00          
1024               | 40960.00         | 5120.00          | 640.00          
2048               | 81920.00         | 10240.00         | 1280.00         
4096               | 163840.00        | 20480.00         | 2560.00         
8192               | 327680.00        | 40960.00         | 5120.00         
16384              | 655360.00        | 81920.00         | 10240.00        


### Output Explanation: KV Cache Memory Comparison
- **Linear Scaling**: The KV cache size scales linearly with sequence length $L$.
- **Saving Ratio**: Grouped-Query Attention (8 heads) consumes exactly 8x less VRAM than standard MHA (64 heads). Multi-Query Attention (1 head) consumes 64x less VRAM.


## 5. Simulation of PagedAttention Page Table Mapping

In [5]:
class MockPagedAttentionManager:
    def __init__(self, block_size: int = 4, num_blocks: int = 16):
        self.block_size = block_size
        self.num_blocks = num_blocks
        self.free_blocks = list(range(num_blocks))
        self.page_table = {} # Maps request_id -> List of physical block indices
        
    def allocate_request(self, request_id: int, num_tokens: int):
        needed_blocks = (num_tokens + self.block_size - 1) // self.block_size
        allocated = []
        for _ in range(needed_blocks):
            if not self.free_blocks:
                raise MemoryError("Out of physical VRAM blocks!")
            allocated.append(self.free_blocks.pop(0))
        self.page_table[request_id] = allocated
        print(f"Allocated request {request_id} ({num_tokens} tokens) to physical blocks: {allocated}")
        
    def add_token(self, request_id: int, current_num_tokens: int):
        # Allocate new page block if current block is full
        if current_num_tokens % self.block_size == 0:
            if not self.free_blocks:
                raise MemoryError("Out of physical VRAM blocks!")
            new_block = self.free_blocks.pop(0)
            self.page_table[request_id].append(new_block)
            print(f"Request {request_id} page limit reached. Allocated new physical block: {new_block}. Block list: {self.page_table[request_id]}")
        else:
            last_block = self.page_table[request_id][-1]
            print(f"Request {request_id} token fits in existing block: {last_block}")

# Verify manager execution
manager = MockPagedAttentionManager(block_size=4, num_blocks=10)
# 1. Allocate initial sequence space of 6 tokens (takes 2 blocks)
manager.allocate_request(request_id=42, num_tokens=6)

# 2. Add tokens step-by-step
manager.add_token(request_id=42, current_num_tokens=6) # Fits in second block
manager.add_token(request_id=42, current_num_tokens=7) # Fits in second block
manager.add_token(request_id=42, current_num_tokens=8) # Page block is full! Allocates a third block


Allocated request 42 (6 tokens) to physical blocks: [0, 1]
Request 42 token fits in existing block: 1
Request 42 token fits in existing block: 1
Request 42 page limit reached. Allocated new physical block: 2. Block list: [0, 1, 2]


### Output Explanation: PagedAttention Simulation
- **Block-wise Allocation**: Memory is allocated dynamically in pages of size 4 tokens.
- **No Fragmentation**: When a page is full (at current token 8), a new block is fetched from the shared pool, eliminating the need for contiguous pre-allocated static VRAM arrays.
